In [1]:
%pip install ultralytics opencv-python matplotlib tqdm

Note: you may need to restart the kernel to use updated packages.


  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
You should consider upgrading via the 'c:\Users\klanz\AppData\Local\Programs\Python\Python310\python.exe -m pip install --upgrade pip' command.


In [2]:
from ultralytics import YOLO
from pathlib import Path
from tqdm import tqdm
from pathlib import Path
import cv2
import os

In [ ]:
CHICKEN_PATH = Path(r"..\raw_data\kury") 
NOT_CHICKEN_PATH = Path(r"..\raw_data\niekury")

IMAGE_EXTENSIONS = [
    ".jpg",
    ".jpeg",
    ".png",
    ".bmp"
]

In [6]:
model = YOLO("yolov8x.pt")

In [ ]:
ANIMAL_CLASSES = {
    14,  # bird
    15,  # cat
    16,  # dog
    17,  # horse
    18,  # sheep
    19,  # cow
    20,  # elephant
    21,  # bear
    22,  # zebra
    23,  # giraffe
}

In [8]:
def save_yolo_labels(results, txt_path):

    with open(txt_path, "w") as f:

        for box in results.boxes:

            cls = int(box.cls.item())

            if cls not in ANIMAL_CLASSES:
                continue

            x, y, w, h = box.xywhn[0].tolist()

            f.write(f"{cls} {x:.6f} {y:.6f} {w:.6f} {h:.6f}\n")

In [ ]:
def annotate_folder(folder):

    images = []

    for ext in IMAGE_EXTENSIONS:
        images.extend(folder.glob(f"*{ext}"))

    not_detected = []

    for image_path in tqdm(images):

        txt_path = image_path.with_suffix(".txt")

        if txt_path.exists():
            continue

        results = model.predict(
            str(image_path),
            verbose=False,
            conf=0.25
        )[0]

        animal_boxes = [
            b for b in results.boxes
            if int(b.cls.item()) in ANIMAL_CLASSES
        ]

        if len(animal_boxes) == 0:
            not_detected.append(image_path.name)
            txt_path.touch()
            continue

        results.boxes = animal_boxes

        save_yolo_labels(results, txt_path)

    return not_detected

In [10]:
print("Chicken...")
missing_chicken = annotate_folder(CHICKEN_PATH)

print("Other...")
missing_other = annotate_folder(NOT_CHICKEN_PATH)

Chicken...


100%|██████████| 3357/3357 [02:35<00:00, 21.62it/s]


Other...


100%|██████████| 2461/2461 [01:32<00:00, 26.55it/s]


In [12]:
with open("missing_annotations.txt", "w") as f:

    f.write("CHICKEN\n")

    for img in missing_chicken:
        f.write(img + "\n")

    f.write("\nOTHER\n")

    for img in missing_other:
        f.write(img + "\n")

print("Zapisano missing_annotations.txt")

Zapisano missing_annotations.txt


In [ ]:
CHICKEN_PATH = Path(r"..\raw_data\kury") 
NOT_CHICKEN_PATH = Path(r"..\raw_data\niekury")


CHICKEN_LIST = Path(r"..\notebooks\chickenmissing.txt")
NOT_CHICKEN_LIST = Path(r"..\notebooks\notchickenmissing.txt")

def remove_files(list_file: Path, image_folder: Path):

    removed_images = 0
    removed_labels = 0
    missing_files = 0

    with open(list_file, "r", encoding="utf-8") as f:
        filenames = [line.strip() for line in f if line.strip()]

    for filename in filenames:

        image_path = image_folder / filename
        label_path = image_folder / (Path(filename).stem + ".txt")

        if image_path.exists():
            image_path.unlink()
            removed_images += 1
        else:
            missing_files += 1

        if label_path.exists():
            label_path.unlink()
            removed_labels += 1

    print(f"\nFolder: {image_folder}")
    print(f"  Usunięte obrazy : {removed_images}")
    print(f"  Usunięte etykiety: {removed_labels}")
    print(f"  Brakujących obrazów: {missing_files}")


remove_files(CHICKEN_LIST, CHICKEN_PATH)
remove_files(NOT_CHICKEN_LIST, NOT_CHICKEN_PATH)

print("\nGotowe.")


Folder: C:\Users\klanz\Desktop\MAGISTERKA\raw_data\kury
  Usunięte obrazy : 114
  Usunięte etykiety: 114
  Brakujących obrazów: 0

Folder: C:\Users\klanz\Desktop\MAGISTERKA\raw_data\niekury
  Usunięte obrazy : 124
  Usunięte etykiety: 124
  Brakujących obrazów: 0

Gotowe.


In [ ]:

CHICKEN_PATH = Path(r"..\raw_data\kury") 
NOT_CHICKEN_PATH = Path(r"..\raw_data\niekury")

def change_class(folder, new_class):

    txt_files = list(folder.rglob("*.txt"))

    changed = 0
    empty = 0

    for txt_file in txt_files:

        with open(txt_file, "r") as f:
            lines = f.readlines()

        if len(lines) == 0:
            empty += 1
            continue

        new_lines = []

        for line in lines:

            parts = line.strip().split()

            if len(parts) == 0:
                continue

            parts[0] = str(new_class)

            new_lines.append(" ".join(parts))

        with open(txt_file, "w") as f:
            f.write("\n".join(new_lines))

        changed += 1

    print(f"{folder.name}")
    print(f"zmieniono: {changed}")
    print(f"pustych:   {empty}")


change_class(CHICKEN_PATH, 0)
change_class(NOT_CHICKEN_PATH, 1)

kury
zmieniono: 3243
pustych:   0
niekury
zmieniono: 2294
pustych:   303
